# VectorDrugMatch - treats-edge ablation (GCN message passing), Colab run - single-device GPU rerun

Question: does letting **training-positive** `treats` edges take part in message passing help a GCN on the drug-disjoint benchmark, without target leakage? Design, leakage safeguards and caveats: `Docs/TREATS_EDGE_ABLATION_PLAN.md`.

**Why this rerun.** In the first run (11-12 Nov 2026) 27 of the 42 runs executed on CPU and 15 on a Tesla T4 after a disconnect. The runner enables mixed precision only on GPU, so those runs differ in numerics as well as speed, and `A_k4_L3` (CPU) vs `B_k4_L3` (GPU) straddles the two devices. This notebook repeats **all 42 runs on the T4** and writes them to a **new** folder `VDM_Outputs/treats_edge_ablation_20261112_gpu/`. The first run's folder `VDM_Outputs/treats_edge_ablation_20261111/` is not touched; keep it as the CPU record.

Run the cells **top to bottom, one at a time**. The code is pinned to commit `fee2401c15` on branch `exp/treats-edge-ablation` (unchanged from the first run). Every cell from Step 4 onwards stops with an error if no GPU is attached.

Prerequisite: `MyDrive/VDM_Data/frozen_processed/primekg/` exists (the upload from the main handbook).

## Step 1 - GPU

Runtime > Change runtime type > T4 GPU. The cell must print `GPU available: True`; if it stops with an error, fix the runtime type and run it again.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU attached. Runtime > Change runtime type > T4 GPU, then run this cell again.'
print(torch.cuda.get_device_name(0))
!nvidia-smi | head -15

## Step 2 - Drive, code at the pinned commit, libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf VectorDrugMatch
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = 'fee2401c155e8ed8c50374e3ccf472e0e1ec60c7'
!git checkout $PIN
!git log -1 --oneline

In [ ]:
%pip install -q rdkit mlflow shap xgboost loguru pyyaml torch-geometric tabulate scikit-learn lxml matplotlib pymupdf pyarrow
import torch, torch_geometric
print('PyTorch', torch.__version__, '| PyG', torch_geometric.__version__, '| CUDA', torch.cuda.is_available())
try:
    import pyg_lib; print('pyg-lib', pyg_lib.__version__)
except Exception as e:
    print('pyg-lib not available (only needed for the RGCN configuration):', e)

## Step 3 - Put the frozen benchmark in place and verify it (stop on a mismatch)

In [ ]:
import hashlib, json, os, shutil
D = '/content/drive/MyDrive/VDM_Data/frozen_processed/primekg'
assert os.path.isdir(D), 'upload MyDrive/VDM_Data/frozen_processed/primekg first (see the main handbook)'
shutil.copytree(D, 'data/processed/primekg', dirs_exist_ok=True)
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
want = {f['path']: f['sha256'] for f in man if f['path'].startswith('data/processed/primekg/') and f['path'].endswith(('.csv', '.npy', '.npz'))}
bad = [p for p, h in want.items() if hashlib.sha256(open(p, 'rb').read()).hexdigest() != h]
print(len(want), 'files checked;', 'MISMATCH:' if bad else 'FROZEN BENCHMARK VERIFIED', bad)
assert not bad

## Step 4 - Outputs to Drive (a new folder for the GPU rerun; the first run's folder is not touched)

Re-run this cell after every reconnect (after Steps 1-3): it defines `N`, which every later cell uses.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'No GPU attached. Runtime > Change runtime type > T4 GPU, then rerun Steps 1-4.'
N = 'treats_edge_ablation_20261112_gpu'
O = f'/content/drive/MyDrive/VDM_Outputs/{N}'
os.makedirs(O, exist_ok=True); os.makedirs('outputs', exist_ok=True)
if not os.path.islink(f'outputs/{N}'):
    os.symlink(O, f'outputs/{N}')
print(os.path.realpath(f'outputs/{N}'), '|', torch.cuda.get_device_name(0))

## Step 5 - Graph diagnostics and the automated leakage audit (CPU)Must print `all_pass=True` for the three graph variants. Compare with `reports/treats_edge_ablation_20261111/diagnostics/` (computed locally). Stop on any FAIL.

In [ ]:
!python scripts/graph_diagnostics.py --out outputs/{N}/diagnostics

## Step 5b - Pre-run verification (folds, masking, held-out absence, alternate representations)Must end with `ALL PASS` (the same check passed locally; `reports/treats_edge_ablation_20261111/verification_pre_run.json`).

In [ ]:
!python scripts/verify_ablation_setup.py

## Step 6 - Smoke test (every configuration, seed 42, 3 epochs; written to `smoke/`, never mixed with results)Checks that every configuration runs and passes its leakage audit. Note the seconds it prints: they are the first timing measurement.

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/run_treats_edge_ablation.py --out outputs/{N} --smoke

## Step 7 - Sensitivity of the masking fraction k (validation ROC-AUC only, seed 42)k is the number of folds per epoch: 1/k of the train positives is hidden from message passing in each step. **k = 4 is fixed a priori for the main runs**; this table only shows sensitivity and is not used to pick the final configuration after seeing test scores.

In [ ]:
!python scripts/run_treats_edge_ablation.py --out outputs/{N} --config-name B_k2 B_k4 B_k8 --seeds 42 --resume
import json, glob
for c in ('B_k2', 'B_k4', 'B_k8'):
    for f in glob.glob(f'outputs/{N}/{c}/seed42/metrics.json'):
        print(c, 'val ROC-AUC', round(json.load(open(f))['val']['roc_auc'], 4))

## Step 8 - Main runs, five seeds (21, 42, 52, 72, 100). Resumable: re-run this cell after a disconnect (Steps 1-4 first).Order: the control and the main comparison first.

In [ ]:
!python scripts/run_treats_edge_ablation.py --out outputs/{N} --config-name A_orig A_k4 B_k4 --resume

In [ ]:
!python scripts/run_treats_edge_ablation.py --out outputs/{N} --config-name C_k4 A_k4_L3 B_k4_L3 --resume

In [ ]:
# optional relation-aware comparison and the DIAGNOSTIC naive-inclusion run (target leakage by construction; never report B0 as a result)
!python scripts/run_treats_edge_ablation.py --out outputs/{N} --config-name D_rgcn_B B0_naive --resume

## Step 9 - Check that every run used the GPU, then summarise

Stops if any of the 42 runs is missing or was not run on a GPU (a `--resume` after a reconnect to a CPU runtime would otherwise mix devices again). Then writes `seed_wise_metrics.csv`, `five_seed_summary.csv`, `paired_vs_A_orig.csv` and `README_results.md` into the Drive folder. Report these numbers as they are; see the plan for how to interpret them.

In [ ]:
import json, glob, collections
rc = [json.load(open(f)) for f in sorted(glob.glob(f'outputs/{N}/*/seed*/run_config.json'))]
print(len(rc), 'runs |', dict(collections.Counter(str(r['gpu']) for r in rc)), '|', dict(collections.Counter(r['torch'] for r in rc)))
not_gpu = [(r['config'], r['seed']) for r in rc if not r['gpu']]
assert not not_gpu, f'these runs did not use a GPU; delete their folders in outputs/{N} and rerun them on the T4: {not_gpu}'
assert len(rc) == 42, f'expected 42 runs (8 configurations x 5 seeds + B_k2, B_k8 at seed 42), found {len(rc)}; rerun Steps 7-8'

!python scripts/summarize_treats_ablation.py --root outputs/{N} --reference A_orig
print(open(f'outputs/{N}/README_results.md').read())

## Step 10 - Save everything to Drive and finish the sync

Copies the MLflow database (it exists only on the Colab machine), writes the light zip (everything except `model.pth`), checks the summary files and then flushes Drive. **Wait until it prints `SAVED`** before closing the tab. Afterwards also use File > Save a copy in Drive to keep this notebook with its printed outputs.

In [ ]:
import os, shutil
from google.colab import drive
if os.path.isfile('mlflow.db'):
    shutil.copy2('mlflow.db', f'{O}/mlflow.db')
!cd outputs && rm -f /content/drive/MyDrive/VDM_Outputs/{N}_results_light.zip && zip -rq /content/drive/MyDrive/VDM_Outputs/{N}_results_light.zip {N} -x '*/model.pth' -x '*/mlflow.db'
for f in ('seed_wise_metrics.csv', 'five_seed_summary.csv', 'paired_vs_A_orig.csv', 'README_results.md', 'mlflow.db'):
    print(f, 'OK' if os.path.isfile(f'{O}/{f}') else 'MISSING')
print(sum(len(fs) for _, _, fs in os.walk(O)), 'files in', O)
print('light zip MB:', round(os.path.getsize(f'/content/drive/MyDrive/VDM_Outputs/{N}_results_light.zip') / 1e6, 1))
drive.flush_and_unmount()
print('SAVED - download', N + '_results_light.zip', 'from MyDrive/VDM_Outputs')